In [35]:
import xml.etree.ElementTree as ET

# Load the XML file
tree = ET.parse("Ikegami-3Shift-DATA1.ros")  # Update with your path if needed
root = tree.getroot()

# === BASIC INFO ===
start_date = root.findtext("StartDate")
end_date = root.findtext("EndDate")

# === SKILLS ===
skills = [skill.get("ID") for skill in root.findall(".//Skills/Skill")]

# === SHIFT TYPES ===
shifts = [shift.get("ID") for shift in root.findall(".//ShiftTypes/Shift")]


# === STAFF ===
nurses = [emp.get("ID") for emp in root.findall(".//Employees/Employee")]

# === DAYS ===

from datetime import datetime, timedelta

start_date_str = root.findtext("StartDate")
end_date_str = root.findtext("EndDate")

# Convert string to datetime
start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")
end_dt = datetime.strptime(end_date_str, "%Y-%m-%d")

# Generate list of dates
days = [(start_dt + timedelta(days=i)).date() for i in range((end_dt - start_dt).days + 1)]

# convert to indices
day_indices = list(range(len(days)))

# === COVERAGE ===

coverage_requirements = {}

# Iterate over all <Cover> elements
for cover in root.findall(".//CoverRequirements/Cover"):
    day = int(cover.get("Day"))
    shift = cover.get("ShiftType")
    skill = cover.get("Skill")
    required = int(cover.get("Required"))
    coverage_requirements[(day, shift, skill)] = required

# === SUMMARY  ===
print("Start Date:", start_date)
print("End Date:", end_date)
print("Number of Nurses:", len(nurses))
print("Number of Days:", len(days))
print("Shifts:", shifts)
print("Skills (sample):", skills[:5])
print("Nurses (sample):", nurses[:5])


Start Date: 1995-10-26
End Date: 1995-11-30
Number of Nurses: 25
Number of Days: 36
Shifts: ['N', 'E', 'D', 'O']
Skills (sample): ['GroupA', 'GroupB', '1', '2', '3']
Nurses (sample): ['A01', 'A02', 'A03', 'A04', 'A05']


In [36]:
import gurobipy as gp
from gurobipy import GRB

# Initialize model
model = gp.Model("Nurse_Rostering")

# Decision variables
x = model.addVars(nurses, days, shifts, vtype=GRB.BINARY, name="assign")

# Constraints
# Each nurse works at most one shift per day
for n in nurses:
    for d in days:
        model.addConstr(gp.quicksum(x[n, d, s] for s in shifts) <= 1, name=f"OneShiftPerDay_{n}_{d}")

# Coverage requirements
for (d, s, sk), req in coverage_requirements.items():
    model.addConstr(gp.quicksum(x[n, d, s] for n in nurses) >= req, name=f"Coverage_{d}_{s}_{sk}")

# Objective function: Minimize total penalty (placeholder)

model.setObjective(gp.quicksum(x[n, d, s] for n in nurses for d in days for s in shifts), GRB.MINIMIZE)
model.optimize()


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) Ultra 5 125U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 900 rows, 3600 columns and 3600 nonzeros
Model fingerprint: 0xc261aa93
Variable types: 0 continuous, 3600 integer (3600 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 14 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%


In [37]:
def fix_and_optimize(model, x, nurses, days, shifts, window_size=5, iterations=10):
    for it in range(iterations):
        start_day_idx = (it * window_size) % len(days)
        end_day_idx = start_day_idx + window_size
        window_days = days[start_day_idx:end_day_idx]

        # Unfix all variables
        for n in nurses:
            for d in days:
                for s in shifts:
                    var = x[n, d, s]
                    var.lb = 0
                    var.ub = 1

        # Fix variables outside window
        for n in nurses:
            for d in days:
                if d not in window_days:
                    for s in shifts:
                        var = x[n, d, s]
                        # Fix to current solution if available
                        if var.X is not None:
                            val = var.X
                            var.lb = val
                            var.ub = val
                        else:
                            # If not solved yet, you might set fix to 0 or 1 as initial guess
                            var.lb = 0
                            var.ub = 1

        # Optimize the subproblem
        model.optimize()


In [ ]:
# import matplotlib.pyplot as plt
import time

# # Lists to store results
fao_objectives = []
exact_objectives = []
times = []

# Run FAO
start_time = time.time()
fix_and_optimize(model, x, nurses, days, shifts)
fao_time = time.time() - start_time
fao_objectives.append(model.ObjVal)
times.append(fao_time)

# Run exact optimization
model.reset()
start_time = time.time()
model.optimize()
exact_time = time.time() - start_time
exact_objectives.append(model.ObjVal)
times.append(exact_time)

# # Plotting
# plt.plot(['FAO', 'Exact'], [fao_objectives[0], exact_objectives[0]], marker='o')
# plt.ylabel('Objective Value')
# plt.title('FAO vs Exact Optimization')
# plt.show()
